# Phase 2 — Step 5: final A2 submission + **LoRMA on B** + **LoRA on A**

**Step 4 results** (A2, full fine-tuning, causal backbone, fold-2 macro-F1):

| head                           | macro-F1  | micro-F1  |
| ------------------------------ | --------- | --------- |
| **BiLSTM**                     | **0.506** | **0.613** |
| linear                         | 0.497     | 0.593     |
| lookahead (t..t+4)             | 0.470     | 0.593     |
| linear + random window offsets | 0.449     | 0.564     |

→ A2 + BiLSTM head is our best experiment-1 model. Random offsets _hurt_ (RESP 0.22 → 0.03): shifted windows often
lose the document start, where the parties are named — so APP/RESP need the header (evidence for "header memory").

**This notebook (≈ 35–45 min, all cached):**

1. **A2 + BiLSTM, Stage B** → `submission.zip` = Phase-2 submission 2.
2. **B (prefix-LM) + LoRMA** (compulsory): the Phase-1 B model is frozen; only small multiplicative adapters
   `h = (I + (α/r)·B·A)·W₀x` (r = 8) + the head are trained. B was pretrained to read its prefix **bidirectionally**,
   so we try both bidirectional and causal attention and submit the better one → submission 3.
3. **A (plain decoder) + LoRA**: frozen backbone + additive adapters `h = W₀x + (α/r)·B·A·x` (r = 8), causal → submission 4.

Adapters go into every block's `qkv`, `proj` (attention) and `w1, w2, w3` (MLP). Adapter learning rate 5e-4, up to 30 epochs.


## 0. Setup


In [1]:
import sys, importlib
from pathlib import Path
import torch

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / ".gitignore").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
import ner_utils as nu, ner_models as nm, ner_experiment as ne
for m in (nu, nm, ne):
    importlib.reload(m)

def free(exp):
    del exp
    torch.cuda.empty_cache()

c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Experiment 1 final: A2 + full fine-tuning + BiLSTM head → Stage B → submission 2

Stage A is already done (cached). Stage B retrains on fold 1 + fold 2 for the best epoch count (9) and writes the zip.


In [2]:
exp = ne.Experiment(PROJECT_ROOT, {"head": "bilstm"})
zip_a2 = exp.stage_b()
exp.log()
print("UPLOAD:", zip_a2)
free(exp)

experiment: A2_full_causal_bilstm_norm_hm0 | device cuda | precision torch.bfloat16
parameters: 71.91M total, 71.91M trainable | loaded {'arch': 'A2', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 5, 'dropped_groups': ['copy_gate', 'copy_k', 'copy_q', 'lm_head']}
[cache] training 'fold1' already done -> loaded best_fold1.pt (best epoch 9)
Stage A: best epoch 9 | val macro-F1 0.5057 | 0s


epoch  1 | train loss 0.7547 | 72s


epoch  2 | train loss 0.0820 | 71s


epoch  3 | train loss 0.0426 | 77s


epoch  4 | train loss 0.0215 | 66s


epoch  5 | train loss 0.0103 | 67s


epoch  6 | train loss 0.0055 | 67s


epoch  7 | train loss 0.0029 | 65s


epoch  8 | train loss 0.0018 | 67s


epoch  9 | train loss 0.0014 | 67s
training 'full' finished -> loaded best epoch 9


submission OK: c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase2\A2_full_causal_bilstm_norm_hm0\submission\submission.zip | 35 documents | 4,145 entities
test entities per label: {'STAT': 890, 'DATE': 718, 'PREC': 714, 'COURT': 572, 'CASENO': 325, 'AUTH': 283, 'APP': 198, 'RESP': 125, 'A.COUNSEL': 110, 'JUDGE': 91, 'R.COUNSEL': 70, 'WIT': 49}
[cache] already logged
UPLOAD: c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase2\A2_full_causal_bilstm_norm_hm0\submission\submission.zip


## 2. Experiment 2: B + LoRMA (bidirectional vs causal attention)


In [3]:
B_RUN = "B_prefixlm_d640_L10_sc70tr30_200M"
b_results = {}
for causal in (False, True):
    exp = ne.Experiment(PROJECT_ROOT, {"phase1_run": B_RUN, "method": "lorma", "causal": causal, "head": "bilstm"})
    exp.stage_a()
    sc = exp.report(n_examples=0)
    exp.log()
    b_results[causal] = (sc["macro_f1"], exp.name)
    free(exp)
    print("=" * 100)
best_causal = max(b_results, key=lambda c: b_results[c][0])
print("B + LoRMA:", {("causal" if c else "bidirectional"): round(v[0], 4) for c, v in b_results.items()},
      "-> submit", "causal" if best_causal else "bidirectional")
exp = ne.Experiment(PROJECT_ROOT, {"phase1_run": B_RUN, "method": "lorma", "causal": best_causal, "head": "bilstm"})
zip_b = exp.stage_b()
print("UPLOAD:", zip_b)
free(exp)

experiment: B_lormar8_bidir_bilstm_norm_hm0 | device cuda | precision torch.bfloat16
parameters: 72.97M total, 2.92M trainable | loaded {'arch': 'B_prefixlm', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 1, 'dropped_groups': ['lm_head'], 'adapter_params': 1064960}


epoch  1 | train loss 1.6743 | 34s | val macro-F1 0.0000 micro-F1 0.0000  <- best


epoch  2 | train loss 0.2450 | 34s | val macro-F1 0.1515 micro-F1 0.3313  <- best


epoch  3 | train loss 0.1210 | 33s | val macro-F1 0.3564 micro-F1 0.5479  <- best


epoch  4 | train loss 0.0837 | 34s | val macro-F1 0.4163 micro-F1 0.5859  <- best


epoch  5 | train loss 0.0628 | 33s | val macro-F1 0.4229 micro-F1 0.5806  <- best


... (40 outputs left out)

epoch 11 | train loss 0.0042 | 54s


epoch 12 | train loss 0.0033 | 54s


epoch 13 | train loss 0.0028 | 54s
training 'full' finished -> loaded best epoch 13


submission OK: c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase2\B_lormar8_causal_bilstm_norm_hm0\submission\submission.zip | 35 documents | 3,855 entities
test entities per label: {'STAT': 791, 'DATE': 736, 'PREC': 646, 'COURT': 552, 'CASENO': 298, 'AUTH': 285, 'APP': 106, 'JUDGE': 104, 'R.COUNSEL': 100, 'RESP': 85, 'WIT': 77, 'A.COUNSEL': 75}
UPLOAD: c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase2\B_lormar8_causal_bilstm_norm_hm0\submission\submission.zip


## 3. Experiment 3: A + LoRA (causal, BiLSTM head)


In [4]:
A_RUN = "A_decoder_d640_L10_sc70tr30_200M"
exp = ne.Experiment(PROJECT_ROOT, {"phase1_run": A_RUN, "method": "lora", "causal": True, "head": "bilstm"})
exp.stage_a()
exp.report(n_examples=0)
exp.log()
zip_a = exp.stage_b()
print("UPLOAD:", zip_a)
free(exp)

experiment: A_lorar8_causal_bilstm_norm_hm0 | device cuda | precision torch.bfloat16
parameters: 72.78M total, 2.73M trainable | loaded {'arch': 'A_decoder', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 1, 'dropped_groups': ['lm_head'], 'adapter_params': 875520}


epoch  1 | train loss 1.8849 | 27s | val macro-F1 0.0002 micro-F1 0.0005  <- best


epoch  2 | train loss 0.2446 | 27s | val macro-F1 0.1322 micro-F1 0.2978  <- best


epoch  3 | train loss 0.1209 | 27s | val macro-F1 0.2550 micro-F1 0.4439  <- best


epoch  4 | train loss 0.0808 | 27s | val macro-F1 0.4195 micro-F1 0.5863  <- best


epoch  5 | train loss 0.0536 | 28s | val macro-F1 0.4885 micro-F1 0.6198  <- best


... (28 outputs left out)

epoch 13 | train loss 0.0017 | 54s


epoch 14 | train loss 0.0014 | 55s


epoch 15 | train loss 0.0013 | 54s
training 'full' finished -> loaded best epoch 15


submission OK: c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase2\A_lorar8_causal_bilstm_norm_hm0\submission\submission.zip | 35 documents | 3,875 entities
test entities per label: {'STAT': 858, 'DATE': 735, 'PREC': 651, 'COURT': 545, 'CASENO': 305, 'AUTH': 264, 'APP': 106, 'JUDGE': 100, 'RESP': 99, 'A.COUNSEL': 86, 'R.COUNSEL': 77, 'WIT': 49}
UPLOAD: c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\checkpoints\phase2\A_lorar8_causal_bilstm_norm_hm0\submission\submission.zip


## 4. All Phase-2 runs so far (fold-2 validation)


In [5]:
rows = ne.compare(PROJECT_ROOT)

run                                         macro  micro  ep     APP   RESP A.COUN R.COUN  JUDGE    WIT   AUTH  COURT   STAT   PREC   DATE CASENO
A_lorar8_causal_bilstm_norm_hm0            0.5092 0.6215  15   0.206  0.177  0.357  0.547  0.634  0.366  0.437  0.816  0.735  0.408  0.946  0.480
A2_full_causal_bilstm_norm_hm0             0.5057 0.6133   9   0.208  0.185  0.454  0.505  0.690  0.245  0.361  0.814  0.708  0.476  0.948  0.473
B_lormar8_causal_bilstm_norm_hm0           0.5034 0.6164  13   0.235  0.115  0.438  0.532  0.548  0.335  0.439  0.824  0.724  0.390  0.933  0.528
A2_full_causal_norm_hm0                    0.4975 0.5934   4   0.249  0.215  0.494  0.503  0.651  0.232  0.389  0.784  0.676  0.429  0.941  0.407
B_lormar8_bidir_bilstm_norm_hm0            0.4845 0.6006  10   0.202  0.098  0.271  0.565  0.608  0.281  0.431  0.812  0.711  0.417  0.934  0.485
A2_full_causal_look4_norm_hm0              0.4698 0.5928  12   0.141  0.162  0.335  0.464  0.591  0.245  0.373  0.810  0.672

## What to do after it finishes

Upload the three zips printed as `UPLOAD:` (A2 BiLSTM, B LoRMA, A LoRA) to Codabench, then tell me "done" + the scores when they appear.
